In [1]:
!pip install opencv-python tensorflow keras torch torchvision tqdm
!pip install gdown


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!pip install kaggle


In [4]:
from google.colab import files
files.upload()


Saving kaggle (1).json to kaggle (1).json


{'kaggle (1).json': b'{"username":"hafidabouftou","key":"b63c20d6df9f5fbc5377b52128a13c37"}'}

In [6]:
!mv "kaggle (1).json" kaggle.json

In [7]:
!pip install kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [13]:
!kaggle datasets download -d odins0n/ucf-crime-dataset -p /content/drive/MyDrive/UCF_Crime --unzip


Dataset URL: https://www.kaggle.com/datasets/odins0n/ucf-crime-dataset
License(s): CC0-1.0
^C


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [9]:
!ls /content/drive/MyDrive/UCF_Crime

ls: cannot access '/content/drive/MyDrive/UCF_Crime': No such file or directory


Étape 1 — Prétraitement + segmentation

In [10]:
import cv2
import numpy as np

def load_video(video_path, resize=(112,112)):
    cap = cv2.VideoCapture(video_path)
    frames = []

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, resize)
        frames.append(frame)

    cap.release()
    return np.array(frames)

def segment_video(frames, num_segments=32):
    T = len(frames)
    seg_len = T // num_segments
    segments = []

    for i in range(num_segments):
        start = i * seg_len
        end = start + seg_len
        seg = frames[start:end]
        segments.append(seg)

    return segments


Étape 1 — Extraction features C3D (simplifiée

In [11]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_c3d_like():
    inp = layers.Input(shape=(16,112,112,3))
    x = layers.Conv3D(32, 3, activation='relu')(inp)
    x = layers.MaxPool3D((2,2,2))(x)
    x = layers.Conv3D(64, 3, activation='relu')(x)
    x = layers.GlobalAveragePooling3D()(x)
    out = layers.Dense(256, activation='relu')(x)
    return models.Model(inp, out)

c3d_model = build_c3d_like()

def extract_c3d_features(segments):
    feats = []
    for seg in segments:
        if len(seg) < 16:
            continue
        clip = seg[:16]
        clip = clip / 255.0
        clip = np.expand_dims(clip, axis=0)
        f = c3d_model.predict(clip, verbose=0)
        feats.append(f[0])
    return np.array(feats)


Étape 2 — Dueling DQN

In [ ]:
def build_dueling_dqn(state_size, action_size):
    inp = layers.Input(shape=(state_size,))
    x = layers.Dense(256, activation='relu')(inp)
    x = layers.Dense(128, activation='relu')(x)

    v = layers.Dense(64, activation='relu')(x)
    v = layers.Dense(1)(v)

    a = layers.Dense(64, activation='relu')(x)
    a = layers.Dense(action_size)(a)

    q = v + (a - tf.reduce_mean(a, axis=1, keepdims=True))

    model = models.Model(inp, q)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss='mse')
    return model


Étape 2 — Prioritized Replay (simplifié)

In [ ]:
import random

class PERBuffer:
    def __init__(self, size=5000, alpha=0.6):
        self.buffer = []
        self.priorities = []
        self.size = size
        self.alpha = alpha

    def add(self, experience, td_error):
        p = (abs(td_error) + 1e-5) ** self.alpha
        if len(self.buffer) >= self.size:
            self.buffer.pop(0)
            self.priorities.pop(0)
        self.buffer.append(experience)
        self.priorities.append(p)

    def sample(self, batch_size):
        probs = np.array(self.priorities)
        probs /= probs.sum()
        idxs = np.random.choice(len(self.buffer), batch_size, p=probs)
        return [self.buffer[i] for i in idxs]


Agent RL

In [ ]:
class DQNAgent:
    def __init__(self, state_size, action_size):
        self.state_size = state_size
        self.action_size = action_size
        self.gamma = 0.95
        self.epsilon = 1.0
        self.epsilon_decay = 0.995
        self.epsilon_min = 0.05

        self.model = build_dueling_dqn(state_size, action_size)
        self.buffer = PERBuffer()

    def act(self, state):
        if np.random.rand() < self.epsilon:
            return random.randrange(self.action_size)
        q = self.model.predict(state[np.newaxis], verbose=0)[0]
        return np.argmax(q)

    def remember(self, s, a, r, s2, done):
        target = r
        if not done:
            target += self.gamma * np.max(self.model.predict(s2[np.newaxis], verbose=0)[0])
        q_val = self.model.predict(s[np.newaxis], verbose=0)[0][a]
        td_error = target - q_val
        self.buffer.add((s,a,r,s2,done), td_error)

    def replay(self, batch_size=32):
        batch = self.buffer.sample(batch_size)
        for s,a,r,s2,done in batch:
            target = r
            if not done:
                target += self.gamma * np.max(self.model.predict(s2[np.newaxis], verbose=0)[0])
            q_vals = self.model.predict(s[np.newaxis], verbose=0)
            q_vals[0][a] = target
            self.model.fit(s[np.newaxis], q_vals, epochs=1, verbose=0)

        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay


Étape 3 — Entraînement

In [ ]:
import os

base = "ucf_crime/Train"
videos = []
labels = []

for cls in os.listdir(base):
    folder = os.path.join(base, cls)
    for f in os.listdir(folder):
        if f.endswith(".mp4"):
            videos.append(os.path.join(folder, f))
            labels.append(0 if cls=="Normal" else 1)

agent = None

for vid, lab in zip(videos[:15], labels[:15]):  # petit subset
    frames = load_video(vid)
    segments = segment_video(frames)
    feats = extract_c3d_features(segments)

    env = VideoEnv(feats, lab)
    state = env.reset()

    if agent is None:
        agent = DQNAgent(len(state), feats.shape[0])

    done = False
    while not done:
        a = agent.act(state)
        s2, r, done = env.step(a)
        agent.remember(state, a, r, s2, done)
        state = s2

    agent.replay(32)
    print("trained on:", vid)


Étape 4 — Test

In [ ]:
def test_video(video_path, agent):
    frames = load_video(video_path)
    segments = segment_video(frames)
    feats = extract_c3d_features(segments)

    env = VideoEnv(feats, label=1)
    state = env.reset()
    done = False

    kept = []

    while not done:
        a = agent.act(state)
        kept.append(a)
        state, _, done = env.step(a)

    score = len(kept) / 32
    return score
